### RUN IN CPU ~00:05:00

In [1]:
# CELL 1
import time

start_time = time.time()

In [2]:
# CELL 2: Setup
import sys, os
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    rl_root = Path("/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2")
    !pip install -q pyarrow fastparquet
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.8 MB/s eta 0:00:00


In [3]:
# CELL 3: Load Data & Align Dates
import pandas as pd
import numpy as np
from core.settings import TradingConfig
from core.paths import GLOBAL_PROCESSED_DIR, LOCAL_DATA_DIR
from data_pipeline.loader import load_raw_global_data
from data_pipeline.utils import get_master_trading_calendar
from data_pipeline.builder import generate_features

pd.set_option("future.no_silent_downcasting", True)
config = TradingConfig()

print("Loading raw global data...")
df_ohlcv, df_fed, df_indices = load_raw_global_data()

print(f"Aligning to Master Trading Calendar ({config.calendar_ticker})...")
master_dates = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

# Align df_ohlcv
full_multi_index = pd.MultiIndex.from_product(
    [df_ohlcv.index.get_level_values("Ticker").unique(), master_dates],
    names=["Ticker", "Date"],
)
df_ohlcv = df_ohlcv.reindex(full_multi_index)
price_cols = ["Adj Open", "Adj High", "Adj Low", "Adj Close"]
if config.handle_zeros_as_nan:
    df_ohlcv[price_cols] = df_ohlcv[price_cols].replace(0, np.nan)
df_ohlcv[price_cols] = df_ohlcv.groupby(level="Ticker")[price_cols].ffill(
    limit=config.max_data_gap_ffill
)
df_ohlcv["Volume"] = df_ohlcv["Volume"].fillna(0)

# Align df_fed & df_indices
if df_fed is not None:
    df_fed["Date"] = pd.to_datetime(df_fed["Date"])
    df_fed = df_fed.set_index("Date").reindex(master_dates).ffill()
if df_indices is not None:
    df_indices = df_indices[
        df_indices.index.get_level_values("Date").isin(master_dates)
    ].sort_index(level=[0, "Date"])

NOTEBOOKS_RLVR_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2
PROJECT_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks
GLOBAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data
GLOBAL_PROCESSED_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data/processed
LOCAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data
OUTPUT_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output

Loading raw global data...
Aligning to Master Trading Calendar (SPY)...


In [4]:
# CELL 4: Generate Features
print("Generating features_df and macro_df (~6 mins)...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv, df_indices=df_indices, df_fed=df_fed, config=config
)

Generating features_df and macro_df (~6 mins)...
[EXEC] Generating Decoupled Features (Benchmark: SPY)...


In [5]:
# CELL 5: Save Datasets
print("Saving Processed Datasets...")
df_ohlcv.to_parquet(GLOBAL_PROCESSED_DIR / "df_ohlcv.parquet")
if df_fed is not None:
    df_fed.to_parquet(GLOBAL_PROCESSED_DIR / "df_fed.parquet")
if df_indices is not None:
    df_indices.to_parquet(GLOBAL_PROCESSED_DIR / "df_indices.parquet")

features_df.to_parquet(LOCAL_DATA_DIR / "features_df.parquet")
macro_df.to_parquet(LOCAL_DATA_DIR / "macro_df.parquet")
print("All exports complete.")

Saving Processed Datasets...
All exports complete.


In [6]:
# CELL 6: Timing
end_time = time.time()
print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}")

Time: 00:05:15
